# Preparación del Dataset: Bike Sharing Demand

## 1. Descripción del Dataset
Datos de demanda de alquiler de bicicletas urbanas basadas en condiciones climáticas (temperatura, viento) y tipos de días (festivos).

Aquí procesaremos y prepararemos los datos matemáticamente desde cero usando fórmulas puras, tal como se hace en el curso. Explicaremos paso a paso el código.


## 2. Importación de Librerías
Vamos a cargar las herramientas necesarias, usando lo más básico posible:

- `pandas` y `numpy`: para leer el archivo y manipular la tabla matemáticamente.
- `matplotlib.pyplot` y `seaborn`: para hacer gráficos simples en 2D (líneas, barras), nada en 3D para no complicarlo.

*Nota:* No usaremos funciones complejas o cajas negras para normalizar ni para dividir los datos, todo lo haremos "a mano" con simples matemáticas.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# NO usamos sklearn para transformar ni dividir datos.


## 3. Cargar y Explorar el Archivo (Dataset)
Aquí leemos el archivo base y mostramos cómo se ve. Revisamos la cantidad de filas y columnas que tenemos para entender su tamaño.


In [2]:
ruta = 'REPOSITORIOS_PARCIAL/4_Bike Sharing  Bike Rental (UCI  Kaggle)/bike-sharing-demand/train.csv'

tabla_datos = pd.read_csv(ruta)

# Mostramos tamaño (filas, columnas)
print('Tamaño del dataset:', tabla_datos.shape)

# Vemos las primeras 5 filas
tabla_datos.head()

# Imprimir información detallada del dataset
filas, columnas = tabla_datos.shape
print(f"\n--- INFORMACIÓN DEL DATASET ---")
print(f"Total de Filas (Ejemplos): {filas}")
print(f"Total de Columnas (Características): {columnas}\n")


Tamaño del dataset: (10886, 12)


,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count
0,2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0,3,13,16
1,2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0,8,32,40
2,2011-01-01 02:00:00,1,0,0,1,9.02,13.635,80,0.0,5,27,32
3,2011-01-01 03:00:00,1,0,0,1,9.84,14.395,75,0.0,3,10,13
4,2011-01-01 04:00:00,1,0,0,1,9.84,14.395,75,0.0,0,1,1


## 4. Análisis de Datos Vacíos o Faltantes
En la vida real, los datos no siempre vienen perfectos, hay casillas vacías (Nulos o NaN). 

En este caso, hemos decidido **eliminar la fila completa** si detectamos que tiene algún dato faltante. Esto asegura que la red neuronal solo aprenda de casos reales y 100% verídicos (no inventamos promedios ni asumimos valores vacíos), a costa de reducir la cantidad de ejemplos disponibles.


In [3]:
datos_nulos = tabla_datos.isnull().sum()
print('Columnas con casillas vacías antes de limpieza:')
print(datos_nulos[datos_nulos > 0])

# Eliminamos absolutamente cualquier fila que contenga un dato nulo (NaN)
filas_antes = len(tabla_datos)
tabla_datos = tabla_datos.dropna()
filas_despues = len(tabla_datos)

print(f"\nSe eliminaron {filas_antes - filas_despues} filas que contenían datos nulos.")
print('Nulos restantes sumados:', tabla_datos.isnull().sum().sum())


Columnas con casillas vacías:
Series([], dtype: int64)

Nulos restantes sumados: 0


C:\Users\alfao\AppData\Local\Temp\ipykernel_18464\885227427.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columnas_texto = tabla_datos.select_dtypes(include=['object']).columns


## 5. Conversión de Textos a Números
Como la computadora o nuestra Red Neuronal no entiende palabras , necesitamos números. Usaremos una herramienta integrada muy sencilla llamada `get_dummies`.

Lo que hace es tomar las columnas de texto y crear columnas visuales llenas de ceros (`0`) y unos (`1`). Es directo y no necesitas complicarlo con diccionarios o For loops.


In [4]:
# Convertimos todo el texto a datos de 0 y 1 de forma directa y sencilla.
tabla_datos = pd.get_dummies(tabla_datos, dtype=int)

# Mostramos cómo quedó la tabla ahora, lista solo con formato numérico
tabla_datos.head()


,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,...,datetime_2012-12-19 14:00:00,datetime_2012-12-19 15:00:00,datetime_2012-12-19 16:00:00,datetime_2012-12-19 17:00:00,datetime_2012-12-19 18:00:00,datetime_2012-12-19 19:00:00,datetime_2012-12-19 20:00:00,datetime_2012-12-19 21:00:00,datetime_2012-12-19 22:00:00,datetime_2012-12-19 23:00:00
0,1,0,0,1,9.84,14.395,81,0.0,3,13,...,0,0,0,0,0,0,0,0,0,0
1,1,0,0,1,9.02,13.635,80,0.0,8,32,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,1,9.02,13.635,80,0.0,5,27,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,1,9.84,14.395,75,0.0,3,10,...,0,0,0,0,0,0,0,0,0,0
4,1,0,0,1,9.84,14.395,75,0.0,0,1,...,0,0,0,0,0,0,0,0,0,0


## 6. Normalización de características (Feature Normalize)
Al visualizar los datos se puede observar que las características tienen diferentes magnitudes (por ejemplo, precios altísimos y números pequeños).

**La fórmula para calcular esto con escalado Z-Score es:**
$$ X_{norm} = \frac{X - \mu}{\sigma} $$

En código lo aplicamos usando las operaciones directas `mean()` (media) y `std()` (desviación estándar).


In [5]:
# 1. Calculamos la media (mu) y desviación estándar (sigma) para cada una de las columnas
mu = tabla_datos.mean()
sigma = tabla_datos.std()

# Para evitar errores por dividir entre cero (si la desviación es 0 en alguna columna), forzamos que sea mínimo 1
sigma_seguro = np.where(sigma == 0, 1, sigma)

# 2. Aplicamos la fórmula matemática exacta: (X - mu) / sigma
datos_normalizados = (tabla_datos - mu) / sigma_seguro

# Mostrar la tabla final convertida
datos_normalizados.head()


,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,...,datetime_2012-12-19 14:00:00,datetime_2012-12-19 15:00:00,datetime_2012-12-19 16:00:00,datetime_2012-12-19 17:00:00,datetime_2012-12-19 18:00:00,datetime_2012-12-19 19:00:00,datetime_2012-12-19 20:00:00,datetime_2012-12-19 21:00:00,datetime_2012-12-19 22:00:00,datetime_2012-12-19 23:00:00
0,-1.349802,-0.171483,-1.460605,-0.660148,-1.333599,-1.092687,0.993167,-1.567682,-0.660962,-0.943810,...,-0.009584,-0.009584,-0.009584,-0.009584,-0.009584,-0.009584,-0.009584,-0.009584,-0.009584,-0.009584
1,-1.349802,-0.171483,-1.460605,-0.660148,-1.438841,-1.182367,0.941206,-1.567682,-0.560882,-0.818015,...,-0.009584,-0.009584,-0.009584,-0.009584,-0.009584,-0.009584,-0.009584,-0.009584,-0.009584,-0.009584
2,-1.349802,-0.171483,-1.460605,-0.660148,-1.438841,-1.182367,0.941206,-1.567682,-0.620930,-0.851119,...,-0.009584,-0.009584,-0.009584,-0.009584,-0.009584,-0.009584,-0.009584,-0.009584,-0.009584,-0.009584
3,-1.349802,-0.171483,-1.460605,-0.660148,-1.333599,-1.092687,0.681399,-1.567682,-0.660962,-0.963673,...,-0.009584,-0.009584,-0.009584,-0.009584,-0.009584,-0.009584,-0.009584,-0.009584,-0.009584,-0.009584
4,-1.349802,-0.171483,-1.460605,-0.660148,-1.333599,-1.092687,0.681399,-1.567682,-0.721009,-1.023260,...,-0.009584,-0.009584,-0.009584,-0.009584,-0.009584,-0.009584,-0.009584,-0.009584,-0.009584,-0.009584


## 7. Dividir el Dataset en 80/20 (Entrenamiento y Prueba)
Finalmente apartaremos el 80% de nuestros datos para que el modelo aprenda y un 20% para hacerle un examen y evaluarlo.
En lugar de una caja negra, apartamos las filas simplemente cortando las tablas basándonos en el porcentaje.


In [6]:
# Extraemos la última columna para usarla como objetivo de predicción (Variable Y o Target)
objetivo_col = datos_normalizados.columns[-1]
caracteristicas = datos_normalizados.drop(objetivo_col, axis=1)
objetivo = datos_normalizados[objetivo_col]

# --------- DIVISIÓN A MANO (80% Entrenamiento, 20% Prueba) ---------

# 1. Calculamos cuánto es el 80% entero de nuestras filas
total_filas = len(datos_normalizados)
ochenta_por_ciento = int(total_filas * 0.8)

# 2. Cortamos las tablas usando posiciones desde 0 hasta el 80%
X_entrenamiento = caracteristicas.iloc[:ochenta_por_ciento]
y_entrenamiento = objetivo.iloc[:ochenta_por_ciento]

# 3. Y para el resto (el 20% restante), usamos desde el 80% en adelante
X_prueba = caracteristicas.iloc[ochenta_por_ciento:]
y_prueba = objetivo.iloc[ochenta_por_ciento:]

print('Total datos Original:', total_filas, 'filas')
print('Datos para Entrenar (80%):', len(X_entrenamiento), 'filas')
print('Datos para Probar (20%):', len(X_prueba), 'filas')
print('\n¡El Dataset está preparado puramente usando lógica matemática!')


Total datos Original: 10886 filas
Datos para Entrenar (80%): 8708 filas
Datos para Probar (20%): 2178 filas

¡El Dataset está preparado puramente usando lógica matemática!
